# Import libraries

In [ ]:
from gulpPP import utils as utils
from gulpPP import ImagingPreProc as iPP
from gulpPP import ROIs as ROIs
import matplotlib.pyplot as plt
import numpy as np
import pickle

# Get file names for a given set of experiments

In [ ]:
file_names = utils.loadFileNames()

# For each tif:
## 1. Create motion corrected MIPs
## 2. Specify the ROIs
## 3. Calculate the DF/F

### Specify parameters for motion correction

In [ ]:
numRefImg = 100 # the number of images to average for the reference image
upsampleFactor = 20 # how much to upsample the image in order to shift the image by less than one pixel
sigma = 2 # the sigma to use in Gaussian filtering

In [ ]:
for i,file in enumerate(file_names):
    # Load the tif
    [stack, nCh, nDiscardFBFrames, fpv] = iPP.loadTif(file)

    # Plot the mean of each plane
    fig_mean_planes = iPP.plotMeanPlane(stack,col=0) # col=1 to plot the second channel if it exists
    
    # Specify the planes to use for the Maximum Intensity Projection (MIP)
    slices = iPP.getSlicesFromStack()
    
    # Calculate the MIP
    stack_MIP = iPP.stackToMIP(stack,slices)
    
    # Motion correct the MIP
    locRefImg = round(stack_MIP.shape[0]/12)# the initial position in the stack to use for the reference
    [shift, stack_MC] = iPP.tifMotionCorrect(numRefImg, locRefImg, upsampleFactor, stack_MIP, sigma)
    
    # Plot the before and after
    fig, axs = plt.subplots(ncols = 2, nrows = 1, figsize = (6,2))
    axs[0].imshow(stack_MIP.mean(axis=0))
    axs[0].axis('off')
    axs[1].imshow(stack_MC.mean(axis=0))
    axs[1].axis('off')
    
    # Get the ROIS - For EB wedges, there are a number of other possible ROI functions for different shapes
    if i == 0:
        [EBOutline, allROIs, allMasks] = iPP.getROIs(stack_MC.mean(axis=0), ROIs.WedgeROIs, [],'')
    else:
        [EBOutline, allROIs, allMasks] = iPP.getROIs(stack_MC.mean(axis=0), ROIs.WedgeROIs, EBOutline, 'Ellipse')
    
    # Get the raw fluorescence
    rawF_G = iPP.FfromROIs(stack_MC, allMasks)
    
    # Get the DF/F
    DF_G = iPP.DFoF(rawF_G)

    # Put all of the data into an object
    exptDat = {'trialName':file.split('/')[-1][0:-4],
               'meanMIP_G': np.squeeze(stack_MC.mean(axis=0)),
               'ROIOutlines': EBOutline,
               'allROIs': allROIs,
               'rawF_G':rawF_G,
               'DF_G': DF_G
              }

    # Pickle and save the data
    outfile = open(file[0:-4] + '_DF.p', 'wb')
    pickle.dump(exptDat, outfile)
    outfile.close()